# Lesson 10: The Grand Market Analysis - Capstone Project

---

## The Final Briefing

It's your final day as a Spark analyst at **PropInsight Analytics**.
The board has requested a **complete market intelligence report** before the quarterly meeting.
They want everything: market overview, agent performance, neighborhood rankings,
mortgage rate trends, and — most importantly — a list of **investment opportunities**.

You have exactly one tool: Apache Spark. And exactly 10 days of experience.
That's all you need.

---

## The 10-Day Journey — A Quick Recap

| Lesson | Topic | Key skill |
|---|---|---|
| 1 | Hello Spark | SparkSession, DataFrames, show() |
| 2 | Reading Data | CSV, JSON, Parquet, schemas |
| 3 | Filtering & Selecting | where(), select(), col() |
| 4 | Aggregations | groupBy(), agg(), pivot() |
| 5 | Joins | inner, left, broadcast |
| 6 | Window Functions | rank(), lag(), running totals |
| 7 | Spark SQL | SQL queries on DataFrames |
| 8 | Enrichment | String, date functions, UDFs |
| 9 | Pipelines | Writing data, caching, optimization |
| 10 | **Capstone** | **Everything, end-to-end** |

## What We'll Build Today

1. **Phase 1** - Data Ingestion: Load all 4 datasets with validation
2. **Phase 2** - Data Quality: Null audit, deduplication
3. **Phase 3** - Data Enrichment: Price per sqft, age categories, days on market
4. **Phase 4** - Master Dataset: Join everything into one wide table
5. **Phase 5** - Market Summary: Key metrics and price analysis
6. **Phase 6** - Agent Leaderboard: Top performers ranked by commissions
7. **Phase 7** - Neighborhood Scorecard: Multi-metric neighborhood ranking
8. **Phase 8** - Mortgage Rate Trends: Best time to buy analysis
9. **Phase 9** - Investment Signals: Undervalued property discovery
10. **Delivery**: Write all outputs to disk

Let's build the report.


In [1]:
# ============================================================
# ALL IMPORTS - production-ready setup
# ============================================================
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    # Column selection & transformation
    col, lit, when, coalesce,
    # String functions
    upper, lower, trim, length, concat_ws, regexp_replace,
    # Numeric functions
    round as spark_round, abs as spark_abs,
    # Aggregation functions
    avg, count, sum as spark_sum, min as spark_min, max as spark_max,
    countDistinct, first,
    # Date functions
    to_date, year, month, date_format, datediff, current_date,
    # Window functions
    rank, dense_rank, row_number, lag,
    # Join helpers
    broadcast,
    # Null handling
    isnull, isnan
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType
)

# ============================================================
# SPARK SESSION - production configuration
# ============================================================
spark = SparkSession.builder \
    .appName("Lesson10_GrandMarketAnalysis") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10485760") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("All systems ready!")
print(f"Spark version:           {spark.version}")
print(f"Shuffle partitions:      {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Broadcast threshold:     {int(spark.conf.get('spark.sql.autoBroadcastJoinThreshold')) / 1024 / 1024:.0f} MB")
print()
print("PropInsight Analytics - Grand Market Analysis")
print("=" * 50)

All systems ready!
Spark version:           4.1.1


Shuffle partitions:      4
Broadcast threshold:     10 MB

PropInsight Analytics - Grand Market Analysis


## Phase 1: Data Ingestion

The first step in any real pipeline is loading data with **explicit schemas**.
Relying on `inferSchema=True` in production is risky — a single malformed row
can cause Spark to infer the wrong type for an entire column.

Best practice: define your schema explicitly and let Spark validate incoming data against it.


In [2]:
# Define explicit schemas for CSV files
listings_schema = StructType([
    StructField("property_id",   StringType(),  nullable=True),
    StructField("address",       StringType(),  nullable=True),
    StructField("city",          StringType(),  nullable=True),
    StructField("zip_code",      StringType(),  nullable=True),
    StructField("property_type", StringType(),  nullable=True),
    StructField("bedrooms",      IntegerType(), nullable=True),
    StructField("bathrooms",     DoubleType(),  nullable=True),
    StructField("sqft",          IntegerType(), nullable=True),
    StructField("list_price",    DoubleType(),  nullable=True),
    StructField("year_built",    IntegerType(), nullable=True),
    StructField("neighborhood",  StringType(),  nullable=True),
    StructField("status",        StringType(),  nullable=True),
    StructField("listing_date",  StringType(),  nullable=True),
    StructField("agent_id",      StringType(),  nullable=True),
])

mortgage_schema = StructType([
    StructField("date",             StringType(), nullable=True),
    StructField("rate_30yr_fixed",  DoubleType(), nullable=True),
    StructField("rate_15yr_fixed",  DoubleType(), nullable=True),
    StructField("rate_5_1_arm",     DoubleType(), nullable=True),
])

# Load all 4 datasets
listings_raw = spark.read \
    .option("header", "true") \
    .schema(listings_schema) \
    .csv("data/property_listings.csv")

commissions_raw = spark.read \
    .option("multiline", "true") \
    .json("data/agent_commissions.json")

neighborhood_stats = spark.read.parquet("data/neighborhood_stats.parquet")

mortgage_rates_raw = spark.read \
    .option("header", "true") \
    .schema(mortgage_schema) \
    .csv("data/mortgage_rates.csv")

# Validate record counts
listings_count     = listings_raw.count()
commissions_count  = commissions_raw.count()
neighborhood_count = neighborhood_stats.count()
mortgage_count     = mortgage_rates_raw.count()

print("Data Ingestion Summary")
print("-" * 40)
print(f"  property_listings.csv    : {listings_count:>6,} records")
print(f"  agent_commissions.json   : {commissions_count:>6,} records")
print(f"  neighborhood_stats.parq  : {neighborhood_count:>6,} records")
print(f"  mortgage_rates.csv       : {mortgage_count:>6,} records")
print("-" * 40)
print(f"  Total records loaded     : {listings_count + commissions_count + neighborhood_count + mortgage_count:>6,}")

Data Ingestion Summary
----------------------------------------
  property_listings.csv    :    510 records
  agent_commissions.json   :    363 records
  neighborhood_stats.parq  :      8 records
  mortgage_rates.csv       :    731 records
----------------------------------------
  Total records loaded     :  1,612


## Phase 2: Data Quality Report

Before any analysis, you must understand the health of your data.
A data quality report answers:
- Which columns have nulls, and how many?
- Are there duplicate records?
- Are there outliers or obviously wrong values?

Fixing these issues before analysis prevents misleading results.


In [3]:
# Count nulls per column in listings
from pyspark.sql.functions import sum as spark_sum

print("NULL COUNT REPORT - property_listings")
print("-" * 50)

null_counts = listings_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in listings_raw.columns
])

# Transpose for readable display
null_rows = null_counts.collect()[0].asDict()
for column_name, null_count in sorted(null_rows.items(), key=lambda x: -x[1]):
    pct = null_count / listings_count * 100
    flag = " <-- ATTENTION" if pct > 5 else ""
    print(f"  {column_name:<20}: {null_count:>4} nulls ({pct:>5.1f}%){flag}")

print()

# Check for duplicate property_ids
total_rows   = listings_raw.count()
distinct_ids = listings_raw.select("property_id").distinct().count()
duplicates   = total_rows - distinct_ids

print("DEDUPLICATION REPORT")
print("-" * 50)
print(f"  Total rows:            {total_rows:,}")
print(f"  Distinct property_ids: {distinct_ids:,}")
print(f"  Duplicate rows:        {duplicates:,}")

# Remove duplicates (keep first occurrence)
listings_deduped = listings_raw.dropDuplicates(["property_id"])
print(f"  After deduplication:   {listings_deduped.count():,} rows")

# Drop rows where critical fields are null
listings_clean = listings_deduped.dropna(subset=["list_price", "sqft", "neighborhood"])
print(f"  After dropping nulls in critical fields: {listings_clean.count():,} rows")

NULL COUNT REPORT - property_listings
--------------------------------------------------


  sqft                :  510 nulls (100.0%) <-- ATTENTION
  year_built          :  510 nulls (100.0%) <-- ATTENTION
  listing_date        :   66 nulls ( 12.9%) <-- ATTENTION
  agent_id            :   50 nulls (  9.8%) <-- ATTENTION
  bathrooms           :   39 nulls (  7.6%) <-- ATTENTION
  property_type       :   31 nulls (  6.1%) <-- ATTENTION
  list_price          :   29 nulls (  5.7%) <-- ATTENTION
  status              :   26 nulls (  5.1%) <-- ATTENTION
  neighborhood        :   23 nulls (  4.5%)
  bedrooms            :   20 nulls (  3.9%)
  property_id         :    0 nulls (  0.0%)
  address             :    0 nulls (  0.0%)
  city                :    0 nulls (  0.0%)
  zip_code            :    0 nulls (  0.0%)



DEDUPLICATION REPORT
--------------------------------------------------
  Total rows:            510
  Distinct property_ids: 500
  Duplicate rows:        10


  After deduplication:   500 rows


  After dropping nulls in critical fields: 0 rows


## Phase 3: Data Enrichment

Clean data becomes *insightful* data through enrichment.
We'll add derived columns that answer questions the raw data cannot:
- How much does this property cost per square foot?
- Is this a new build or a vintage home?
- How long has it been sitting on the market?


In [4]:
CURRENT_YEAR = 2024

listings_enriched = listings_clean \
    .withColumn(
        "price_per_sqft",
        spark_round(col("list_price") / col("sqft"), 2)
    ) \
    .withColumn(
        "property_age",
        lit(CURRENT_YEAR) - col("year_built")
    ) \
    .withColumn(
        "age_category",
        when(col("property_age") < 5,  "Brand New")
        .when(col("property_age") < 20, "Modern")
        .when(col("property_age") < 40, "Established")
        .otherwise("Vintage")
    ) \
    .withColumn(
        "price_category",
        when(col("list_price") < 300000,  "Budget")
        .when(col("list_price") < 600000,  "Mid-Range")
        .when(col("list_price") < 1000000, "Premium")
        .otherwise("Luxury")
    ) \
    .withColumn(
        "listing_date_parsed",
        to_date(col("listing_date"), "yyyy-MM-dd")
    ) \
    .withColumn(
        "days_on_market",
        datediff(current_date(), col("listing_date_parsed"))
    ) \
    .withColumn(
        "neighborhood", trim(col("neighborhood"))
    ) \
    .withColumn(
        "property_type", upper(trim(col("property_type")))
    )

print("Enriched listings schema:")
listings_enriched.printSchema()

print("\nSample enriched records:")
listings_enriched.select(
    "property_id", "list_price", "sqft", "price_per_sqft",
    "year_built", "property_age", "age_category",
    "price_category", "days_on_market"
).show(5)

Enriched listings schema:
root
 |-- property_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- sqft: integer (nullable = true)
 |-- list_price: double (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- neighborhood: string (nullable = true)
 |-- status: string (nullable = true)
 |-- listing_date: string (nullable = true)
 |-- agent_id: string (nullable = true)
 |-- price_per_sqft: double (nullable = true)
 |-- property_age: integer (nullable = true)
 |-- age_category: string (nullable = false)
 |-- price_category: string (nullable = false)
 |-- listing_date_parsed: date (nullable = true)
 |-- days_on_market: integer (nullable = true)


Sample enriched records:


+-----------+----------+----+--------------+----------+------------+------------+--------------+--------------+
|property_id|list_price|sqft|price_per_sqft|year_built|property_age|age_category|price_category|days_on_market|
+-----------+----------+----+--------------+----------+------------+------------+--------------+--------------+
+-----------+----------+----+--------------+----------+------------+------------+--------------+--------------+



## Phase 4: Building the Master Dataset

The master dataset is the foundation of the entire report. It joins:

1. **Enriched listings** (the core table)
2. **Neighborhood stats** (broadcast join — only 8 rows)
3. **Commission aggregates** (left join — not all listings have a commission record)

The result is a single wide table with everything in one place.


In [5]:
# Aggregate commissions per property (a property may have multiple commission records)
commission_by_property = commissions_raw.groupBy("property_id").agg(
    spark_sum("commission_amount").alias("total_commission"),
    spark_sum("sale_price").alias("sale_price"),
    avg("commission_rate").alias("avg_commission_rate"),
    count("*").alias("commission_transactions")
)

print(f"Commission aggregates computed for {commission_by_property.count():,} properties")

# Join 1: Listings + Neighborhood stats (broadcast — tiny table)
master = listings_enriched.join(
    broadcast(neighborhood_stats),
    on="neighborhood",
    how="left"
)

# Join 2: Master + Commission aggregates (left join — not all listings are sold)
master = master.join(
    commission_by_property,
    on="property_id",
    how="left"
)

# Cache the master dataset — we'll use it in many downstream queries
master.cache()
master_count = master.count()  # triggers caching

print(f"\nMaster dataset: {master_count:,} rows x {len(master.columns)} columns")
print("\nMaster dataset schema:")
master.printSchema()

print("\nSample master record (first listing):")
master.select(
    "property_id", "address", "neighborhood", "list_price",
    "price_per_sqft", "age_category", "school_rating",
    "crime_rate_per_1000", "total_commission"
).show(3, truncate=False)

Commission aggregates computed for 265 properties



Master dataset: 0 rows x 30 columns

Master dataset schema:
root
 |-- property_id: string (nullable = true)
 |-- neighborhood: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- sqft: integer (nullable = true)
 |-- list_price: double (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- listing_date: string (nullable = true)
 |-- agent_id: string (nullable = true)
 |-- price_per_sqft: double (nullable = true)
 |-- property_age: integer (nullable = true)
 |-- age_category: string (nullable = false)
 |-- price_category: string (nullable = false)
 |-- listing_date_parsed: date (nullable = true)
 |-- days_on_market: integer (nullable = true)
 |-- median_income: long (nullable = true)
 |-- school_rating: double (nullable = 

+-----------+-------+------------+----------+--------------+------------+-------------+-------------------+----------------+
|property_id|address|neighborhood|list_price|price_per_sqft|age_category|school_rating|crime_rate_per_1000|total_commission|
+-----------+-------+------------+----------+--------------+------------+-------------+-------------------+----------------+
+-----------+-------+------------+----------+--------------+------------+-------------+-------------------+----------------+



## Phase 5: Market Summary

The board's first question is always: **"How big is the market?"**

We'll answer with a clean, formatted print report covering:
- Total listings and market value
- Average price overall and by property type
- Average price by neighborhood
- Listing status breakdown


In [6]:
# High-level market metrics
market_metrics = master.agg(
    count("*").alias("total_listings"),
    spark_sum("list_price").alias("total_market_value"),
    avg("list_price").alias("avg_list_price"),
    avg("price_per_sqft").alias("avg_price_per_sqft"),
    avg("days_on_market").alias("avg_days_on_market"),
    avg("sqft").alias("avg_sqft")
).collect()[0]

print("=" * 60)
print(" PROPINSIGHT ANALYTICS - MARKET SUMMARY REPORT")
print("=" * 60)
print(f"  Total Listings:          {int(market_metrics['total_listings']):>10,}")
print(f"  Total Market Value:      ${(market_metrics['total_market_value'] or 0):>14,.0f}")
print(f"  Average List Price:      ${(market_metrics['avg_list_price'] or 0):>14,.0f}")
print(f"  Average Price / Sqft:    ${(market_metrics['avg_price_per_sqft'] or 0):>14,.2f}")
print(f"  Average Sqft:            {(market_metrics['avg_sqft'] or 0):>14,.0f} sqft")
print(f"  Avg Days on Market:      {(market_metrics['avg_days_on_market'] or 0):>14,.1f} days")
print()

# Average price by property type
print("Average List Price by Property Type:")
print("-" * 45)
price_by_type = master.groupBy("property_type").agg(
    count("*").alias("listings"),
    spark_round(avg("list_price"), 0).alias("avg_price"),
    spark_round(avg("price_per_sqft"), 2).alias("avg_ppsf")
).orderBy(col("avg_price").desc())
price_by_type.show(10, truncate=False)

# Average price by neighborhood
print("Average List Price by Neighborhood:")
print("-" * 45)
price_by_neighborhood = master.groupBy("neighborhood").agg(
    count("*").alias("listings"),
    spark_round(avg("list_price"), 0).alias("avg_price")
).orderBy(col("avg_price").desc())
price_by_neighborhood.show(10, truncate=False)

# Listing status breakdown
print("Listing Status Breakdown:")
print("-" * 45)
master.groupBy("status").count().orderBy(col("count").desc()).show()

 PROPINSIGHT ANALYTICS - MARKET SUMMARY REPORT
  Total Listings:                   0
  Total Market Value:      $             0
  Average List Price:      $             0
  Average Price / Sqft:    $          0.00
  Average Sqft:                         0 sqft
  Avg Days on Market:                 0.0 days

Average List Price by Property Type:
---------------------------------------------


+-------------+--------+---------+--------+
|property_type|listings|avg_price|avg_ppsf|
+-------------+--------+---------+--------+
+-------------+--------+---------+--------+

Average List Price by Neighborhood:
---------------------------------------------
+------------+--------+---------+
|neighborhood|listings|avg_price|
+------------+--------+---------+
+------------+--------+---------+

Listing Status Breakdown:
---------------------------------------------


+------+-----+
|status|count|
+------+-----+
+------+-----+



## Phase 6: Agent Performance Leaderboard

Which agents are driving the most revenue? The board wants the top 10 ranked.
We'll use **window functions** to rank agents by total commission earned — the same
technique you learned in Lesson 6.


In [7]:
# Aggregate commission data per agent
agent_performance = commissions_raw.groupBy("agent_id").agg(
    count("*").alias("total_sales"),
    spark_round(spark_sum("commission_amount"), 2).alias("total_commission"),
    spark_round(avg("commission_amount"), 2).alias("avg_commission_per_sale"),
    spark_round(spark_sum("sale_price"), 0).alias("total_sales_volume"),
    spark_round(avg("sale_price"), 0).alias("avg_sale_price")
)

# Rank agents by total commission using a window function
agent_window = Window.orderBy(col("total_commission").desc())

agent_leaderboard = agent_performance \
    .withColumn("rank", rank().over(agent_window)) \
    .withColumn("commission_rank", dense_rank().over(agent_window)) \
    .orderBy("rank")

print("=" * 70)
print(" AGENT PERFORMANCE LEADERBOARD - TOP 10")
print("=" * 70)
agent_leaderboard.select(
    "rank",
    "agent_id",
    "total_sales",
    "total_commission",
    "avg_commission_per_sale",
    "total_sales_volume",
    "avg_sale_price"
).show(10, truncate=False)

# Show the sales volume from commission data alongside listings data
top_agent = agent_leaderboard.first()
print(f"\nTop Agent: {top_agent['agent_id']}")
print(f"  Total Commission Earned: ${top_agent['total_commission']:>12,.2f}")
print(f"  Total Sales Closed:      {top_agent['total_sales']:>12,}")
print(f"  Total Sales Volume:      ${top_agent['total_sales_volume']:>12,.0f}")
print(f"  Average Commission/Sale: ${top_agent['avg_commission_per_sale']:>12,.2f}")

 AGENT PERFORMANCE LEADERBOARD - TOP 10


+----+--------+-----------+----------------+-----------------------+------------------+--------------+
|rank|agent_id|total_sales|total_commission|avg_commission_per_sale|total_sales_volume|avg_sale_price|
+----+--------+-----------+----------------+-----------------------+------------------+--------------+
|1   |AGT-018 |29         |538127.88       |18556.13               |20151991          |694896.0      |
|2   |AGT-009 |24         |534266.89       |22261.12               |18555984          |773166.0      |
|3   |AGT-011 |28         |516840.76       |18458.6                |18412108          |657575.0      |
|4   |AGT-017 |28         |501293.53       |17903.34               |17982162          |642220.0      |
|5   |AGT-005 |21         |474970.36       |22617.64               |16627775          |791799.0      |
|6   |AGT-008 |26         |468205.65       |18007.91               |17720203          |681546.0      |
|7   |AGT-015 |26         |460229.74       |17701.14               |15257


Top Agent: AGT-018
  Total Commission Earned: $  538,127.88
  Total Sales Closed:                29
  Total Sales Volume:      $  20,151,991
  Average Commission/Sale: $   18,556.13


## Phase 7: Neighborhood Scorecard

Not all neighborhoods are created equal. The board wants a **composite score** for each
neighborhood that combines:
- School quality (higher is better)
- Crime rate (lower is better)
- Median income (higher is better)
- Average listing price relative to market (moderate is investable)

This is a classic **multi-criteria scoring** problem — and window functions make the ranking trivial.


In [8]:
# Step 1: Compute average listing price per neighborhood from master dataset
avg_price_per_neighborhood = master.groupBy("neighborhood").agg(
    spark_round(avg("list_price"), 0).alias("avg_listing_price"),
    count("*").alias("listing_count")
)

# Step 2: Join with neighborhood_stats
neighborhood_scorecard = neighborhood_stats.join(
    avg_price_per_neighborhood,
    on="neighborhood",
    how="left"
)

# Step 3: Build a composite score (0-100)
# Normalize each metric: higher school_rating -> higher score
#                        lower crime_rate     -> higher score
#                        higher median_income -> higher score

# We'll use window-based percentile ranking for fair normalization
score_window = Window.orderBy(lit(1))  # dummy window over all rows

# School score: school_rating is 1-10, normalize to 0-40 points
# Crime score:  invert (low crime = high score), scale to 0-35 points
# Income score: normalize to 0-25 points

# Get min/max for normalization
stats = neighborhood_scorecard.agg(
    spark_min("school_rating").alias("min_school"),
    spark_max("school_rating").alias("max_school"),
    spark_min("crime_rate_per_1000").alias("min_crime"),
    spark_max("crime_rate_per_1000").alias("max_crime"),
    spark_min("median_income").alias("min_income"),
    spark_max("median_income").alias("max_income")
).collect()[0]

school_range = stats["max_school"] - stats["min_school"] or 1
crime_range  = stats["max_crime"]  - stats["min_crime"]  or 1
income_range = stats["max_income"] - stats["min_income"] or 1

neighborhood_scored = neighborhood_scorecard \
    .withColumn(
        "school_score",
        spark_round(
            (col("school_rating") - lit(stats["min_school"])) / lit(school_range) * 40, 1
        )
    ) \
    .withColumn(
        "safety_score",
        spark_round(
            (lit(1.0) - (col("crime_rate_per_1000") - lit(stats["min_crime"])) / lit(crime_range)) * 35, 1
        )
    ) \
    .withColumn(
        "income_score",
        spark_round(
            (col("median_income") - lit(stats["min_income"])) / lit(income_range) * 25, 1
        )
    ) \
    .withColumn(
        "composite_score",
        spark_round(col("school_score") + col("safety_score") + col("income_score"), 1)
    )

# Rank neighborhoods by composite score
rank_window = Window.orderBy(col("composite_score").desc())
neighborhood_scored = neighborhood_scored.withColumn("overall_rank", rank().over(rank_window))

print("=" * 80)
print(" NEIGHBORHOOD SCORECARD (scored out of 100)")
print("=" * 80)
neighborhood_scored.select(
    "overall_rank",
    "neighborhood",
    "school_rating",
    "crime_rate_per_1000",
    "median_income",
    "avg_listing_price",
    "school_score",
    "safety_score",
    "income_score",
    "composite_score"
).orderBy("overall_rank").show(truncate=False)

 NEIGHBORHOOD SCORECARD (scored out of 100)


+------------+------------+-------------+-------------------+-------------+-----------------+------------+------------+------------+---------------+
|overall_rank|neighborhood|school_rating|crime_rate_per_1000|median_income|avg_listing_price|school_score|safety_score|income_score|composite_score|
+------------+------------+-------------+-------------------+-------------+-----------------+------------+------------+------------+---------------+
|1           |Parkview    |9.3          |9.4                |99323        |NULL             |34.4        |34.1        |5.8         |74.3           |
|2           |Oakwood     |8.1          |9.57               |131556       |NULL             |21.1        |33.7        |19.1        |73.9           |
|3           |Seaside     |8.2          |9.02               |95320        |NULL             |22.2        |35.0        |4.2         |61.4           |
|4           |Downtown    |7.6          |16.51              |145967       |NULL             |15.6        |

## Phase 8: Mortgage Rate Analysis

Property price is only half the picture. The **mortgage rate** determines the true
monthly cost for a buyer.

We'll analyse the historical rate data to find:
- Monthly average rates
- The best month to lock in a 30-year fixed mortgage
- Recent trend direction (are rates rising or falling?)


In [9]:
# Parse dates and extract month/year
mortgage_rates = mortgage_rates_raw \
    .withColumn("date_parsed", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("yr",  year(col("date_parsed"))) \
    .withColumn("mo",  month(col("date_parsed"))) \
    .withColumn("month_label", date_format(col("date_parsed"), "yyyy-MM"))

# Monthly averages
monthly_rates = mortgage_rates.groupBy("yr", "mo", "month_label").agg(
    spark_round(avg("rate_30yr_fixed"), 3).alias("avg_30yr"),
    spark_round(avg("rate_15yr_fixed"), 3).alias("avg_15yr"),
    spark_round(avg("rate_5_1_arm"),    3).alias("avg_arm")
).orderBy("yr", "mo")

print("Monthly Mortgage Rate Averages (sample - most recent 12 months):")
monthly_rates.orderBy(col("yr").desc(), col("mo").desc()).show(12, truncate=False)

# Find the best month to buy (lowest 30-year fixed rate)
best_month = monthly_rates.orderBy(col("avg_30yr").asc()).first()
worst_month = monthly_rates.orderBy(col("avg_30yr").desc()).first()

print("RATE ANALYSIS")
print("-" * 50)
print(f"  Best month to lock a 30yr rate:  {best_month['month_label']}  ({best_month['avg_30yr']}%)")
print(f"  Worst month (highest rate):      {worst_month['month_label']}  ({worst_month['avg_30yr']}%)")
print(f"  Rate swing across history:       {worst_month['avg_30yr'] - best_month['avg_30yr']:.3f} percentage points")

# Show rate trend using lag() — compare each month to the previous month
rate_window = Window.orderBy("yr", "mo")
rate_trend = monthly_rates \
    .withColumn("prev_30yr", lag("avg_30yr", 1).over(rate_window)) \
    .withColumn("rate_change", spark_round(col("avg_30yr") - col("prev_30yr"), 3)) \
    .withColumn(
        "trend",
        when(col("rate_change") > 0.05, "Rising")
        .when(col("rate_change") < -0.05, "Falling")
        .otherwise("Stable")
    )

print("\nRate trend (most recent 10 months):")
rate_trend.select("month_label", "avg_30yr", "rate_change", "trend") \
    .orderBy(col("yr").desc(), col("mo").desc()) \
    .show(10, truncate=False)

Monthly Mortgage Rate Averages (sample - most recent 12 months):


+----+---+-----------+--------+--------+-------+
|yr  |mo |month_label|avg_30yr|avg_15yr|avg_arm|
+----+---+-----------+--------+--------+-------+
|2024|12 |2024-12    |6.333   |5.833   |5.533  |
|2024|11 |2024-11    |6.284   |5.784   |5.484  |
|2024|10 |2024-10    |6.248   |5.748   |5.448  |
|2024|9  |2024-09    |6.16    |5.66    |5.36   |
|2024|8  |2024-08    |6.273   |5.773   |5.473  |
|2024|7  |2024-07    |6.538   |6.038   |5.738  |
|2024|6  |2024-06    |6.568   |6.068   |5.768  |
|2024|5  |2024-05    |6.647   |6.147   |5.847  |
|2024|4  |2024-04    |6.765   |6.265   |5.965  |
|2024|3  |2024-03    |6.711   |6.211   |5.911  |
|2024|2  |2024-02    |6.647   |6.147   |5.847  |
|2024|1  |2024-01    |6.54    |6.04    |5.74   |
+----+---+-----------+--------+--------+-------+
only showing top 12 rows


RATE ANALYSIS
--------------------------------------------------
  Best month to lock a 30yr rate:  2024-09  (6.16%)
  Worst month (highest rate):      2023-02  (6.794%)
  Rate swing across history:       0.634 percentage points

Rate trend (most recent 10 months):


+-----------+--------+-----------+-------+
|month_label|avg_30yr|rate_change|trend  |
+-----------+--------+-----------+-------+
|2024-12    |6.333   |0.049      |Stable |
|2024-11    |6.284   |0.036      |Stable |
|2024-10    |6.248   |0.088      |Rising |
|2024-09    |6.16    |-0.113     |Falling|
|2024-08    |6.273   |-0.265     |Falling|
|2024-07    |6.538   |-0.03      |Stable |
|2024-06    |6.568   |-0.079     |Falling|
|2024-05    |6.647   |-0.118     |Falling|
|2024-04    |6.765   |0.054      |Rising |
|2024-03    |6.711   |0.064      |Rising |
+-----------+--------+-----------+-------+
only showing top 10 rows


## Phase 9: Investment Signals - Undervalued Properties

This is the most valuable deliverable in the report.

An **undervalued property** in a **desirable neighborhood** is the holy grail of real estate investment.
We define our criteria:

1. **Priced below neighborhood average** — the property is cheaper than its peers
2. **High school rating** (> 7.5) — families are willing to pay a premium here
3. **Low crime rate** (< 15 per 1,000 residents) — safe, stable area
4. **Status = Active** — still available to buy

Properties meeting all four criteria are flagged as **investment opportunities**.


In [10]:
# Step 1: Compute the average list price per neighborhood
neighborhood_avg_price = master.groupBy("neighborhood").agg(
    avg("list_price").alias("neighborhood_avg_price")
)

# Step 2: Join this back onto the master dataset
master_with_avg = master.join(
    broadcast(neighborhood_avg_price),
    on="neighborhood",
    how="left"
)

# Step 3: Compute price difference vs neighborhood average
master_with_avg = master_with_avg.withColumn(
    "price_vs_neighborhood_avg",
    spark_round(col("list_price") - col("neighborhood_avg_price"), 0)
).withColumn(
    "discount_pct",
    spark_round(
        (col("neighborhood_avg_price") - col("list_price")) / col("neighborhood_avg_price") * 100,
        1
    )
)

# Step 4: Apply investment signal filters
investment_opportunities = master_with_avg.filter(
    (col("list_price") < col("neighborhood_avg_price")) &   # below neighborhood avg
    (col("school_rating") > 7.5) &                          # good schools
    (col("crime_rate_per_1000") < 15) &                     # safe neighborhood
    (col("status") == "Active")                             # still available
).orderBy(col("discount_pct").desc())  # biggest discount first

opportunity_count = investment_opportunities.count()

print("=" * 80)
print(" INVESTMENT OPPORTUNITY REPORT")
print(" Criteria: Below neighborhood avg price + school_rating > 7.5 + crime < 15")
print("=" * 80)
print(f"\n  Properties meeting all criteria: {opportunity_count}")
print()

investment_opportunities.select(
    "property_id",
    "address",
    "neighborhood",
    "property_type",
    "bedrooms",
    "sqft",
    "list_price",
    "neighborhood_avg_price",
    "discount_pct",
    "school_rating",
    "crime_rate_per_1000"
).show(15, truncate=False)

print("\nTop 5 opportunities by discount:")
investment_opportunities.select(
    "property_id", "address", "neighborhood",
    "list_price", "neighborhood_avg_price", "discount_pct"
).show(5, truncate=False)

 INVESTMENT OPPORTUNITY REPORT
 Criteria: Below neighborhood avg price + school_rating > 7.5 + crime < 15

  Properties meeting all criteria: 0



+-----------+-------+------------+-------------+--------+----+----------+----------------------+------------+-------------+-------------------+
|property_id|address|neighborhood|property_type|bedrooms|sqft|list_price|neighborhood_avg_price|discount_pct|school_rating|crime_rate_per_1000|
+-----------+-------+------------+-------------+--------+----+----------+----------------------+------------+-------------+-------------------+
+-----------+-------+------------+-------------+--------+----+----------+----------------------+------------+-------------+-------------------+


Top 5 opportunities by discount:


+-----------+-------+------------+----------+----------------------+------------+
|property_id|address|neighborhood|list_price|neighborhood_avg_price|discount_pct|
+-----------+-------+------------+----------+----------------------+------------+
+-----------+-------+------------+----------+----------------------+------------+



In [11]:
# Write ALL results to the output/ directory
import os

print("Writing final outputs...")
print("-" * 50)

# 1. Master dataset — partitioned by neighborhood for fast downstream reads
master.write \
    .mode("overwrite") \
    .partitionBy("neighborhood") \
    .parquet("output/capstone_master_dataset")
print("  [1/6] output/capstone_master_dataset/ (partitioned by neighborhood)")

# 2. Market summary by property type
price_by_type.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("output/market_summary_by_type")
print("  [2/6] output/market_summary_by_type/")

# 3. Agent leaderboard
agent_leaderboard.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("output/agent_leaderboard")
print("  [3/6] output/agent_leaderboard/")

# 4. Neighborhood scorecard
neighborhood_scored.write \
    .mode("overwrite") \
    .parquet("output/neighborhood_scorecard")
print("  [4/6] output/neighborhood_scorecard/")

# 5. Mortgage rate trends
rate_trend.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("output/mortgage_rate_trends")
print("  [5/6] output/mortgage_rate_trends/")

# 6. Investment opportunities — coalesced to single file for easy sharing
investment_opportunities \
    .coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("output/investment_opportunities")
print("  [6/6] output/investment_opportunities/ (single file)")

# Release cache
master.unpersist()

print()
print("All outputs written successfully.")
print("The board report is ready for delivery.")

Writing final outputs...
--------------------------------------------------


  [1/6] output/capstone_master_dataset/ (partitioned by neighborhood)


  [2/6] output/market_summary_by_type/


  [3/6] output/agent_leaderboard/


  [4/6] output/neighborhood_scorecard/


  [5/6] output/mortgage_rate_trends/


  [6/6] output/investment_opportunities/ (single file)

All outputs written successfully.
The board report is ready for delivery.


---

# CONGRATULATIONS!

## You've Completed the Learn Spark With Me Course!

---

You started 10 lessons ago with a blank notebook and a `SparkSession`.
Today you delivered a **production-grade market intelligence report** from four
raw data sources, entirely in PySpark.

Let's look at how far you've come.

---

## The Complete Spark Toolkit You've Built

### Core Foundations
| Concept | What you can do now |
|---|---|
| SparkSession | Create sessions with tuned configurations |
| DataFrames | Read, inspect, and transform any tabular dataset |
| Lazy Evaluation | Understand why Spark waits until an action is called |
| DAG | Explain how Spark builds a plan before executing |

### Data Reading & Writing
| Concept | What you can do now |
|---|---|
| CSV / JSON / Parquet | Read and write all three formats |
| Explicit schemas | Define `StructType` to enforce data types on ingestion |
| Write modes | Choose `overwrite`, `append`, `ignore`, or `error` appropriately |
| Partitioned writes | Use `partitionBy()` for fast partition-pruned reads |

### Transformations
| Concept | What you can do now |
|---|---|
| Filtering & Selecting | `filter()`, `where()`, `select()`, `drop()` |
| Column expressions | `col()`, `lit()`, `when().otherwise()` |
| String functions | `upper`, `lower`, `trim`, `concat_ws`, `regexp_replace` |
| Date functions | `to_date`, `datediff`, `year`, `month`, `date_format` |
| UDFs | Write Python functions and apply them as Spark columns |

### Aggregations & Analytics
| Concept | What you can do now |
|---|---|
| GroupBy + Agg | `groupBy().agg()` with `avg`, `sum`, `count`, `min`, `max` |
| Window Functions | `rank()`, `dense_rank()`, `lag()`, running totals |
| Joins | `inner`, `left`, `broadcast` — and when to use each |
| Spark SQL | Write SQL queries directly on registered DataFrames |

### Performance & Production
| Concept | What you can do now |
|---|---|
| Caching | `cache()` + first action + `unpersist()` |
| Partitioning | `repartition()` vs `coalesce()` — and when each is right |
| Broadcast joins | `broadcast()` for small lookup tables |
| Shuffle tuning | Set `spark.sql.shuffle.partitions` for your data size |
| Pipeline design | Wrap ETL logic in reusable, testable functions |

---

## What to Learn Next

You've mastered the core Spark API. Here's where the journey continues:

### Spark Streaming
Process real-time data streams — clickstreams, IoT sensors, financial ticks.
The API mirrors the batch DataFrame API, so your existing knowledge transfers directly.
> Explore: `spark.readStream`, Structured Streaming, Kafka integration

### Spark MLlib
Build machine learning pipelines at scale — regression, classification, clustering,
collaborative filtering — all running on distributed data.
> Explore: `pyspark.ml`, `Pipeline`, `VectorAssembler`, `LinearRegression`, `RandomForestClassifier`

### Delta Lake
Add ACID transactions, time travel, and schema enforcement to your Parquet files.
Delta Lake is the standard for reliable data lakes at companies running Spark in production.
> Explore: `delta-spark`, `DeltaTable`, `MERGE INTO`, `DESCRIBE HISTORY`

### Cloud Deployment
Run your Spark jobs on managed clusters with auto-scaling and built-in monitoring:
- **Databricks** — the most popular managed Spark platform; built by the creators of Spark
- **AWS EMR** — Elastic MapReduce with S3 as the data lake
- **Google Dataproc** — Spark on GCP with BigQuery integration
- **Azure Synapse Analytics** — Microsoft's integrated analytics service

### Advanced Optimization
When your jobs handle terabytes, these topics become critical:
- Adaptive Query Execution (AQE) — Spark 3.0+
- Z-ordering and data skipping in Delta Lake
- Salting techniques for skewed joins
- Custom Spark extensions and Catalyst rules

---

## A Final Word

Apache Spark is one of the most powerful tools in the modern data engineer's toolkit.
But like any tool, it's only as good as the person wielding it.

The concepts you've learned in these 10 lessons — lazy evaluation, immutable transformations,
distributed partitioning, columnar storage — are not just Spark concepts.
They are **foundational principles of distributed computing** that will serve you
across every data platform you work with for the rest of your career.

Keep building. Keep questioning why things are fast or slow.
Read the Spark UI. Look at query plans. Profile your jobs.
The engineers who truly master Spark are the ones who go one level deeper than
the API and understand what's happening under the hood.

You have the foundation. The rest is practice.

---

> *"The best data engineer is not the one who knows every function —
> it's the one who understands when to use each one, and why."*

---

**Thank you for learning Spark with Anoop.**
Now go build something real.
